# Introduction
GOAL: To train a model to predict the wild fire risk of properties, using census data as input features and the proximity to fires as a target feature.

It would be cool to add in data regarding local climate.

In [1]:
import census
from census import Census

import censusgeocode as cg
# import pytidycensus as tc

import geopandas as gpd
import json
import os
import numpy as np
import pandas as pd
import seaborn as sns
from datetime import datetime
import sqlalchemy as sql
# import matplotlib.pyplot as plt
# import pandas as pd
# from censusdis.states import ALL_STATES_AND_DC

import load_wildfires
import load_census
import load_properties
import gis
import train


from scipy.stats import randint, uniform
from pathlib import Path

from sklearn.ensemble import RandomForestRegressor 
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.metrics import make_scorer, mean_poisson_deviance, mean_squared_error
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import MinMaxScaler, StandardScaler, PowerTransformer
from xgboost import XGBRegressor
from settings import GIS_DESIRED_COLS, GIS_DEFAULT_CRS, SQL_ENGINE_STR
from sql_funcs import SQL

from sqlalchemy import text





In [2]:
sql_obj = SQL(test=True)

#TODO: Remove this when you're done testing
# try:
#     sql_obj._drop_table('census_cache', True)
# except RuntimeError as e:
#     print("census_cache table did not exist.")
#     pass

# Load Properties

In [3]:
# sql_obj.drop_table('properties_test', confirm=True)
# props._populate_test_props(wildfire_gdf=wildfires.data, sql_obj=sql_obj, quantity=50)

## Select Properties of Interest

Chosing 300000 properties randomly from US addresses. We will join relevant census data to these addresses. This will probably take awhile, so best to run it overnight.

We don't care about the address itself. We add a census identifier called the GEOID which based on the coordinate's state, county, and tract number.

Using a package that makes use of the [US Census Geocoder API](https://www.census.gov/programs-surveys/geography/technical-documentation/complete-technical-documentation/census-geocoder.html), requests can be in batches of 10,000.

https://pypi.org/project/random-address/

In [4]:
props = load_properties.Properties(sql_obj=sql_obj)
props.add_random_properties(100)


'properties_test' table found.
Test mode, cannot add more properties.


In [5]:
# wildfires = load_wildfires.WildfireData(sql_obj=sql_obj)
# props._populate_test_props(wildfire_gdf=wildfires.data, sql_obj=sql_obj, quantity = 40)

In [6]:
properties = props.get_properties_gpd()

properties.tail()

,geoid,block_id,block_grp,tract_id,county_id,state_id,geometry
35,281559501003031,3031,3,950100,155,28,POINT (633888.475 1187169.135)
36,010570200002007,2007,2,20000,57,1,POINT (765348.682 1237556.324)
37,131270003022029,2029,2,302,127,13,POINT (1386048.872 1014189.446)
38,050690020001030,1030,1,2000,69,5,POINT (364246.583 1232187.328)
39,400079517002070,2070,2,951700,7,40,POINT (-433840.76 1521678.199)


# Load Features from US Census

2023 US Census Data

Using an API key, we will use the 'census' Python package to interact with the US Govermnent's census API.

In [7]:
census = load_census.CensusData(sql_obj=sql_obj, year=2023, granularity='county')

census_test
Drop success!
props_census_test
Drop success!


In [8]:
combined_gdf = census.merge_census_info(properties)
combined_gdf.head()


39 geographies needed, 0 cached, 39 to fetch.
Unique geoids: 39.
[0] Cache hit
[1] Cache hit
[2] Cache hit
[3] Cache hit
[4] Cache hit
[5] Cache hit
[6] Cache hit
[7] Cache hit
[8] Cache hit
[9] Cache hit
[10] Cache hit
[12] Cache hit
[13] Cache hit
[14] Cache hit
[15] Cache hit
[16] Cache hit
[17] Cache hit
[18] Cache hit
[19] Cache hit
[20] Cache hit
[21] Cache hit
[22] Cache hit
[23] Cache hit
[24] Cache hit
[25] Cache hit
[26] Cache hit
[27] Cache hit
[28] Cache hit
[29] Cache hit
[30] Cache hit
[31] Cache hit
[32] Cache hit
[33] Cache hit
[34] Cache hit
[35] Cache hit
[36] Cache hit
[37] Cache hit
[38] Cache hit
[39] Cache hit
_read_from_sql() Sucessfully tested.


,geoid,geometry,B25014H_002E,B25014H_003E,B25014C_002E,B25014C_003E,B25014_003E,B25014_012E,B25014_013E,B25014_010E,...,B23025_004E,B23025_005E,B23025_007E,B23025_006E,B25031_002E,B25031_006E,B25031_005E,B25031_004E,B25031_003E,B25031_007E
0,050219502001011,POINT (497898.547 1505110.772),0.316956,0.005969,0.001326,0.000000,0.202001,0.001326,0.000000,0.040621,...,0.534794,0.035778,0.429427,0.000000,0.199129,NaN,0.358170,0.282789,0.159913,NaN
1,410390001002000,POINT (-2044796.683 2612309.953),0.280683,0.006183,0.002676,0.000209,0.161519,0.001943,0.000719,0.048629,...,0.564126,0.041058,0.394004,0.000812,0.126722,0.204791,0.200074,0.161599,0.121137,0.185677
2,160039502001208,POINT (-1618688.606 2614137.291),0.314341,0.003850,0.000770,0.000000,0.241963,0.000000,0.000000,0.014629,...,0.456313,0.032409,0.511278,0.000000,NaN,NaN,0.520983,0.479017,NaN,NaN
3,271051051001050,POINT (22979.835 2300224.998),0.228115,0.002339,0.002159,0.001889,0.193252,0.000045,0.000000,0.034368,...,0.623395,0.016737,0.359625,0.000243,NaN,0.234594,0.222699,0.163612,0.129680,0.249415
4,292294901004082,POINT (310668.091 1595371.694),0.309211,0.006534,0.000098,0.000000,0.202809,0.000195,0.000000,0.023114,...,0.492535,0.021249,0.485661,0.000556,NaN,0.202165,0.165156,0.145015,0.092145,0.395519


# Load Wildfire GIS Data for 2024

We will use point data from the Visible Infrared Imaging Radiometer Suite (VIIRS). A valid alternative is using burn boundary data. There are a few different data sources we could use, but in the interest of (portfolio) simplicity we'll use just the VIIRS.

N:B: May be a good chance to practice using AWS DB storage and retrieval?

In [9]:
wildfires = load_wildfires.WildfireData(sql_obj=sql_obj)


Extracting wildfire data from GIS files.
Loading satellite:  J1V-C2
Loading satellite:  J2V-C2
Loading satellite:  LS
Loading satellite:  M-C61
Loading satellite:  SV-C2


## Create Targets (Wildfire Proximity Score)

Give Each Property a Wildfire Risk Score based on the proximity to wildfires.

TODO: List the various options given for targets.


In [10]:
proximity_features = gis.calc_all_features(combined_gdf, wildfires.data)
targets_features = pd.concat([combined_gdf, proximity_features], axis=1)

In [11]:
targets_features.head()

,geoid,geometry,B25014H_002E,B25014H_003E,B25014C_002E,B25014C_003E,B25014_003E,B25014_012E,B25014_013E,B25014_010E,...,exp_decay_score,fire_count_0_10km,fire_count_10_25km,fire_count_25_50km,fire_count_50_100km,fire_FRP_0_10km,fire_FRP_10_25km,fire_FRP_25_50km,fire_FRP_50_100km,nearest_fire_km
0,050219502001011,POINT (497898.547 1505110.772),0.316956,0.005969,0.001326,0.000000,0.202001,0.001326,0.000000,0.040621,...,3890.270659,27.0,77.0,185.0,649.0,593.358833,2425.753500,5645.875667,19502.100278,1.419717
1,410390001002000,POINT (-2044796.683 2612309.953),0.280683,0.006183,0.002676,0.000209,0.161519,0.001943,0.000719,0.048629,...,921.802960,0.0,22.0,38.0,154.0,0.000000,815.720299,1414.829369,6472.946856,13.478232
2,160039502001208,POINT (-1618688.606 2614137.291),0.314341,0.003850,0.000770,0.000000,0.241963,0.000000,0.000000,0.014629,...,1534.946907,5.0,36.0,36.0,483.0,150.395962,1014.563167,1086.500026,19864.737133,5.204598
3,271051051001050,POINT (22979.835 2300224.998),0.228115,0.002339,0.002159,0.001889,0.193252,0.000045,0.000000,0.034368,...,106.406099,0.0,2.0,10.0,37.0,0.000000,37.595000,163.575000,1191.043333,12.168663
4,292294901004082,POINT (310668.091 1595371.694),0.309211,0.006534,0.000098,0.000000,0.202809,0.000195,0.000000,0.023114,...,196.315655,0.0,3.0,26.0,79.0,0.000000,45.950000,512.633333,1560.479476,11.156251


# Machine Learning Considerations
## Scoring Methods

For the float risk score, we can use Mean Squared Error (MSE) or Root Mean Squared Error (RMSE). Since it's quadratic in difference between observations and predictions deviations, MSE strongly penalizes large misses, which would be expensive for the insurance company.

For the risk category counts, they appear to be Poisson distributed, so a Poisson loss-function is appropriate.

For any classification model with the binned risk categories, we want to make large misses costly (i.e. predicting a 1 when the category is a 10), since these would also be very costly to the insurance company. To be honest, MSE will work here as well, since the categories are just 

# Model Machine Learning


NB: A good chance to make use of AWS compute.


### Split Data into Features/Targets

We use `nearest_fire_km` as the target — the distance in kilometres to the nearest wildfire detection. With the full 300k-property dataset, `exp_decay_score` (which captures both proximity and density of nearby fires) would be a better choice, but the small 119-property test set has nearly zero variance in decay score because all properties are ~360 km from the nearest fire.

All other proximity-derived columns are dropped so the model only sees census features as inputs.

In [12]:
TARGET_COL = "nearest_fire_km"

# All proximity features are derived from the same wildfire data — drop them
# so the model only sees census features as inputs.
proximity_cols = [c for c in proximity_features.columns]
drop_cols = ["geometry", "geoid"] + [c for c in proximity_cols if c != TARGET_COL]

X_train, X_test, y_train, y_test = train.prepare_split(
    targets_features, TARGET_COL, drop_cols
)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Target range: {y_train.min():.4f} – {y_train.max():.4f}")
print(f"Target std:   {y_train.std():.6f}")
print(f"Target mean:  {y_train.mean():.4f}")

Train: (32, 856), Test: (8, 856)
Target range: 1.4197 – 28.3302
Target std:   7.748144
Target mean:  12.6798


In [13]:
# Diagnostic: check if the target has meaningful variance
cv = y_train.std() / y_train.mean() * 100  # coefficient of variation
print(f"Coefficient of variation: {cv:.4f}%")
if cv < 1.0:
    print(
        f"WARNING: Target has near-zero variance (CV={cv:.4f}%). "
        f"All properties are ~{y_train.mean():.1f} km from the nearest fire. "
        f"Models will appear to have perfect accuracy but are not learning meaningful patterns. "
        f"Scale to 300k properties for geographic diversity."
    )

Coefficient of variation: 61.1061%


### Preprocessing

Drop high-NaN columns, remove correlated features, then impute + scale. All steps are fit on training data only to prevent leakage.

In [14]:
print(f"Features before filtering: {X_train.shape[1]}")

# Drop columns with >45% NaN (computed on train)
X_train, X_test = train.drop_high_nan_columns(X_train, X_test, threshold=0.45)
print(f"After NaN filter: {X_train.shape[1]}")

# Drop one of each highly correlated pair (|r| > 0.85)
X_train, X_test = train.drop_correlated_features(X_train, X_test, threshold=0.85)
print(f"After correlation filter: {X_train.shape[1]}")

# Save column names before pipeline converts to ndarray
feature_names = X_train.columns.tolist()

# Impute (KNN, k=5) then scale (StandardScaler)
pipeline = train.build_preprocessing_pipeline(n_neighbors=5)
X_train = pipeline.fit_transform(X_train)
X_test = pipeline.transform(X_test)

print(f"Final feature matrix: {X_train.shape}")

Features before filtering: 856
After NaN filter: 848
After correlation filter: 439
Final feature matrix: (32, 439)


#### RandomForestRegressor


In [15]:
rfr_search = train.train_random_forest(X_train, y_train, n_iter=20, cv=5)
print(f"Best RF params: {rfr_search.best_params_}")

rfr_metrics = train.evaluate_model(rfr_search.best_estimator_, X_train, X_test, y_train, y_test)
print(f"RF Train RMSE: {rfr_metrics['train_rmse']:.8f}")
print(f"RF Test  RMSE: {rfr_metrics['test_rmse']:.8f}")

Fitting 5 folds for each of 20 candidates, totalling 100 fits
Best RF params: {'max_depth': 36, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'min_samples_split': 3, 'n_estimators': 198}
RF Train RMSE: 3.01114744
RF Test  RMSE: 15.21355845


#### XGBoost


In [16]:
xgb_search = train.train_xgboost(X_train, y_train, n_iter=20, cv=5)
print(f"Best XGB params: {xgb_search.best_params_}")

xgb_metrics = train.evaluate_model(xgb_search.best_estimator_, X_train, X_test, y_train, y_test)
print(f"XGB Train RMSE: {xgb_metrics['train_rmse']:.8f}")
print(f"XGB Test  RMSE: {xgb_metrics['test_rmse']:.8f}")

Fitting 5 folds for each of 20 candidates, totalling 100 fits
Best XGB params: {'colsample_bytree': np.float64(0.5673719653104866), 'gamma': np.float64(0.25325297155335186), 'learning_rate': np.float64(0.1808309081446487), 'max_depth': 3, 'min_child_weight': 3, 'n_estimators': 938, 'reg_alpha': np.float64(0.22135470999507967), 'reg_lambda': np.float64(0.8231288929093322), 'subsample': np.float64(0.511839578613541)}
XGB Train RMSE: 0.34756997
XGB Test  RMSE: 16.96100771


#### Extract Feature Weights


In [17]:
# Pick the better model
if xgb_metrics["test_rmse"] <= rfr_metrics["test_rmse"]:
    best_model = xgb_search.best_estimator_
    print("Best model: XGBoost")
else:
    best_model = rfr_search.best_estimator_
    print("Best model: RandomForest")

top_features = train.extract_feature_importance(best_model, feature_names, top_n=10)
print(f"\nTop 10 features:\n{top_features}")

Best model: RandomForest

Top 10 features:
B19001H_017E    0.031943
B01002D_001E    0.022962
B19019_007E     0.016312
B25041_005E     0.014169
B25091_022E     0.013360
B25041_006E     0.012879
B25024_008E     0.012729
B01001C_006E    0.012702
B01001C_005E    0.011974
B25032_006E     0.011878
dtype: float64


In [18]:
model_path = Path("Models") / "best_model.pkl"
train.save_model(best_model, model_path, pipeline=pipeline, feature_names=feature_names)
print(f"Model saved to {model_path}")

Model saved to Models\best_model.pkl


# Conclusion